In [1]:
import os
import json
import datetime
import re
import random
from typing import List, Optional, Dict, Tuple

import numpy as np
import scipy.stats as scipy_stats

from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim, amp
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler, Dataset

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    DataCollatorForLanguageModeling,
)
from tqdm.auto import tqdm

In [2]:

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")

Mounted at /content/drive
Google Drive mounted.


In [ ]:
try:
    import spacy
    _nlp_test = spacy.load(
        "en_core_sci_md",
        disable=["tagger", "parser", "ner", "lemmatizer"],
        config={"components": {"tok2vec": {"model": {"embed": {"include_static_vectors": True}}}}}
    )
    print("SUCCESS: SciSpaCy loaded correctly. The pipeline will use it for sentence splitting.")
    del _nlp_test
except Exception as e:
    print(f"SciSpaCy NOT available here. Reason: {type(e).__name__}: {e}")
    print("The pipeline will automatically fall back to a regex-based sentence splitter instead.")


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [4]:
MODEL_MAP = {
    "bert": "bert-base-uncased",
    "scibert": "allenai/scibert_scivocab_uncased",
    "biobert": "dmis-lab/biobert-base-cased-v1.2",
    "pubmedbert": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    "biolinkbert": "michiyasunaga/BioLinkBERT-base"
}
selected_model = "pubmedbert" # select the PLM
BASE_MODEL = MODEL_MAP[selected_model]

In [5]:
RUN_CONFIG1_FOCAL = True
RUN_CONFIG1_CE = True

In [6]:
DATA_DIR = os.environ.get(
    "DATA_DIR", "/content/drive/MyDrive/Zhi_folders/PubMed/pubmed-rct-master/PubMed_20k_RCT"
)
UNLAB_CSV = os.environ.get("UNLAB_CSV", "/content/drive/MyDrive/pubmed/pubmed_abstracts_extracted_9996.csv")

SAVE_DIR = os.environ.get(
    "SAVE_DIR", f"/content/drive/MyDrive/Zhi_folders/config1_vs_cadsca/{selected_model}"
)
RESULTS_DIR = f"{SAVE_DIR}/results"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [7]:
BATCH_SIZE = 32
MAX_LEN = 128
CONFIG1_MAX_LEN = 256




EPOCHS = 20
LEARNING_RATE = 2e-5
PATIENCE = 7
GRAD_CLIP = 1.0
WARMUP_FRACTION = 0.1
MC_PASSES = 30
CI_CONFIDENCE = 0.90
FOCAL_GAMMA = 2.0

# ---- SimCSE pretraining
SIMCSE_BATCH_SIZE = 128
SIMCSE_EPOCHS = 5
SIMCSE_LR = 3e-5
SIMCSE_TEMP = 0.05
SIMCSE_MAX_LEN = 128

In [13]:
PUBLISHED_CADSCA_JSON_PATH = os.environ.get(
    "PUBLISHED_CADSCA_JSON_PATH_ENV", # Using a proper environment variable name
    os.path.join("/content/drive/MyDrive/Zhi_folders/ca-dsca-results/", "published_cadsca_results.json")
)

In [14]:
def _published_ci(mean: float, std: float, n_passes: int = MC_PASSES, confidence: float = CI_CONFIDENCE):
    t_crit = scipy_stats.t.ppf((1 + confidence) / 2, df=n_passes - 1)
    se = std / np.sqrt(n_passes)
    return float(mean), float(std), float(mean - t_crit * se), float(mean + t_crit * se)


In [15]:
def _published_stats_from_raw(raw: Dict) -> Dict:
      return {
        'micro': _published_ci(*raw['micro']),
        'macro': _published_ci(*raw['macro']),
        'per_class': {
            cls: _published_ci(*raw[cls]) for cls in ['BACKGROUND', 'CONCLUSIONS', 'METHODS', 'OBJECTIVE', 'RESULTS']
        },
        'clf_report': "(published in manuscript -- not recomputed here)",
    }

In [16]:
def load_published_cadsca_results(json_path: str = PUBLISHED_CADSCA_JSON_PATH) -> Dict:

    if not os.path.exists(json_path):
        print(f"NOTE: published CA-DSCA reference file not found at {json_path} -- "
              f"the final comparison table will omit published baseline/CA-DSCA rows.")
        return {}
    with open(json_path, 'r') as f:
        raw_data = json.load(f)
    return {
        plm: {config: _published_stats_from_raw(raw[config]) for config in ('baseline', 'cadsca')}
        for plm, raw in raw_data.items() if not plm.startswith('_')
    }


In [17]:
PUBLISHED_CADSCA_RESULTS = load_published_cadsca_results()

In [18]:
def load_pubmed_with_context(file_path):
    abstracts, current_abs = [], []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('###'):
                if current_abs:
                    abstracts.append(current_abs)
                    current_abs = []
                continue
            try:
                label, sentence = line.split('\t', 1)
                current_abs.append((label, sentence))
            except ValueError:
                continue
    if current_abs:
        abstracts.append(current_abs)

    sentences, labels, prev_sents, next_sents = [], [], [], []
    for abstract in abstracts:
        for i, (label, sentence) in enumerate(abstract):
            sentences.append(sentence)
            labels.append(label)
            prev_sents.append(abstract[i - 1][1] if i > 0 else "")
            next_sents.append(abstract[i + 1][1] if i < len(abstract) - 1 else "")
    return sentences, labels, prev_sents, next_sents


In [19]:
def build_config1_context_string(sentences, prev_sents, next_sents):

    out = []
    for prev, target, nxt in zip(prev_sents, sentences, next_sents):
        out.append(f"{prev} [SEP] {target} [SEP] {nxt}")
    return out



In [20]:
def clean_text_for_plm(text) -> str:
    if text is None:
        return ""
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[\u0000-\u001F\u007F]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

In [ ]:
# NOTE: if running in Colab and en_core_sci_md is not yet installed, run this
# !pip install /path/to/scispacy/en_core_sci_md-0.5.4.tar.gz

In [21]:


try:
    import spacy
    nlp = spacy.load(
        "en_core_sci_md",
        disable=["tagger", "parser", "ner", "lemmatizer"],
        config={"components": {"tok2vec": {"model": {"embed": {"include_static_vectors": True}}}}}
    )
    if "senter" not in nlp.pipe_names:
        nlp.add_pipe("sentencizer")

    def sentence_splitter(text: str) -> List[str]:
        doc = nlp(text)
        return [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 3]

    print("Using SciSpaCy sentence splitter.")
except Exception as e:
    print(f"SciSpaCy not available - using regex splitter. Reason: {type(e).__name__}: {e}")

    def sentence_splitter(text: str) -> List[str]:
        sents = re.split(r'(?<=[.!?])\s+', str(text))
        return [s.strip() for s in sents if len(s.strip().split()) > 3]


SciSpaCy not available - using regex splitter. Reason: OSError: [E050] Can't find model 'en_core_sci_md'. It doesn't seem to be a Python package or a valid path to a data directory.


In [22]:
def load_unlabeled_sentences(csv_path: str, n_sample: int = 5000, seed: int = SEED) -> List[str]:
    import pandas as pd
    resolved_path = os.path.abspath(csv_path)
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"\n\n{'!' * 78}\n"
            f"UNLABELED CSV NOT FOUND -- STOPPING BEFORE ANY TRAINING.\n"
            f"  Looked for: {csv_path}\n"
            f"  Resolved to: {resolved_path}\n\n"
            f"SimCSE pretraining is a required part of Config 1 in this script.\n"
            f"Fix the path (set the UNLAB_CSV environment variable, or edit the\n"
            f"UNLAB_CSV default near the top of this file) and re-run.\n\n"
            f"If you deliberately want to run Config 1 WITHOUT SimCSE pretraining\n"
            f"(a different, valid experiment -- 'context + focal/CE, vanilla\n"
            f"encoder'), do not silence this error -- instead rename that\n"
            f"variant explicitly in the config name so results can never be\n"
            f"mislabeled as SimCSE-based, and skip this function on purpose.\n"
            f"{'!' * 78}\n"
        )
    df = pd.read_csv(csv_path)
    if df.empty:
        raise ValueError(
            f"Unlabeled CSV at {csv_path} was found but is EMPTY. "
            f"Stopping before training rather than silently skipping SimCSE."
        )
    if len(df) > n_sample:
        df = df.sample(n=n_sample, random_state=seed).reset_index(drop=True)
    texts = df["Abstract"].fillna("").astype(str).tolist() if "Abstract" in df.columns else df.iloc[:, 0].astype(str).tolist()
    all_sents = []
    for t in texts:
        sents = sentence_splitter(t)
        all_sents.extend(clean_text_for_plm(s) for s in sents if len(s.strip()) > 0)
    print(f"Collected {len(all_sents)} unlabeled sentences for SimCSE pretraining.")
    return all_sents


In [23]:
def encode_single_stream(sentences, labels_array, tokenizer, max_len: int) -> TensorDataset:
    enc = tokenizer(
        list(sentences), max_length=max_len, padding='max_length',
        truncation=True, return_tensors="pt"
    )
    return TensorDataset(enc['input_ids'], enc['attention_mask'], torch.tensor(labels_array))


In [24]:
def focal_loss(logits: torch.Tensor, targets: torch.Tensor,
                class_weights: Optional[torch.Tensor] = None,
                gamma: float = FOCAL_GAMMA, reduction: str = "mean") -> torch.Tensor:
    """Class-weighted focal loss: L = -w_y * (1 - p_y)^gamma * log(p_y)."""
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()
    target_log_probs = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    target_probs = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    focal_term = (1.0 - target_probs).clamp(min=0.0) ** gamma
    loss = -focal_term * target_log_probs
    if class_weights is not None:
        alpha = class_weights.to(logits.device)[targets]
        loss = loss * alpha
    if reduction == "mean":
        return loss.mean()
    elif reduction == "sum":
        return loss.sum()
    return loss


In [25]:
def make_loss_fn(loss_type: str, class_weights: torch.Tensor):

    if loss_type == "focal":
        return lambda logits, targets: focal_loss(logits, targets, class_weights=class_weights,
                                                    gamma=FOCAL_GAMMA, reduction="mean")
    elif loss_type == "ce":
        return lambda logits, targets: F.cross_entropy(logits, targets, weight=class_weights)
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")


In [26]:
class SimCSEModel(nn.Module):
    def __init__(self, base_model_name: str, dropout: float = 0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        cls = outputs.last_hidden_state[:, 0]
        return self.dropout(cls)


In [27]:
def simcse_loss(e1: torch.Tensor, e2: torch.Tensor, temp: float = SIMCSE_TEMP) -> torch.Tensor:
    device = e1.device
    e1 = F.normalize(e1, dim=1)
    e2 = F.normalize(e2, dim=1)
    logits = torch.matmul(e1, e2.T) / temp
    labels_idx = torch.arange(e1.size(0), device=device)
    return F.cross_entropy(logits, labels_idx)

In [28]:

def get_base_transformer(model: nn.Module) -> nn.Module:
    for attr in ["bert", "roberta", "electra", "base_model", "model", "transformer", "encoder"]:
        if hasattr(model, attr):
            return getattr(model, attr)
    return model


def transfer_encoder_weights(src_state: Dict[str, torch.Tensor], target_model: nn.Module) -> int:
    dest = get_base_transformer(target_model)
    dest_state = dest.state_dict()
    matched = {k: v for k, v in src_state.items() if k in dest_state and v.shape == dest_state[k].shape}
    if matched:
        dest_state.update(matched)
        dest.load_state_dict(dest_state, strict=False)
    return len(matched)

In [29]:
def pretrain_simcse(model: nn.Module, texts: List[str], tokenizer, device: torch.device = DEVICE,
                     epochs: int = SIMCSE_EPOCHS, lr: float = SIMCSE_LR, temp: float = SIMCSE_TEMP,
                     batch_size: int = SIMCSE_BATCH_SIZE, max_len: int = SIMCSE_MAX_LEN):
    if not texts:
        print("No unlabeled sentences available; skipping SimCSE pretraining.")
        return

    class _RawTextDS(Dataset):
        def __init__(self, texts):
            self.texts = texts

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            return self.texts[idx]

    def _collate(batch):
        enc = tokenizer(list(batch), max_length=max_len, padding=True, truncation=True, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"]

    loader = DataLoader(_RawTextDS(texts), batch_size=min(batch_size, len(texts)),
                         shuffle=True, collate_fn=_collate, drop_last=True)

    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scaler = amp.GradScaler() if device.type == "cuda" else None

    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        pbar = tqdm(loader, desc=f"SimCSE epoch {epoch + 1}/{epochs}")
        for input_ids, attention_mask in pbar:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            optimizer.zero_grad(set_to_none=True)

            doubled_ids = torch.cat([input_ids, input_ids], dim=0)
            doubled_mask = torch.cat([attention_mask, attention_mask], dim=0)

            if scaler:
                with amp.autocast(device_type="cuda"):
                    emb = model(input_ids=doubled_ids, attention_mask=doubled_mask)
                    e1, e2 = emb.chunk(2, dim=0)
                    loss = simcse_loss(e1, e2, temp=temp)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                emb = model(input_ids=doubled_ids, attention_mask=doubled_mask)
                e1, e2 = emb.chunk(2, dim=0)
                loss = simcse_loss(e1, e2, temp=temp)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            loss_value = loss.detach().float().item()
            total_loss += loss_value
            n_batches += 1
            pbar.set_postfix({"loss": f"{loss_value:.4f}"})

        print(f"[SimCSE] Epoch {epoch + 1}/{epochs} - avg loss: {total_loss / max(1, n_batches):.4f}")
        if device.type == "cuda":
            torch.cuda.empty_cache()



In [30]:
def train_single_stream(model_builder_fn, num_labels: int, train_loader, val_loader,
                         save_path: str, class_weights: torch.Tensor, loss_type: str = "ce",
                         epochs: int = EPOCHS, lr: float = LEARNING_RATE, patience: int = PATIENCE,
                         encoder_init_state: Optional[Dict[str, torch.Tensor]] = None,
                         label: str = "model") -> float:

    if os.path.exists(save_path):
        print(f"[{label}] Checkpoint already exists at {save_path} -- skipping training, will resume for eval.")
        ckpt = torch.load(save_path, map_location=DEVICE)
        return ckpt.get("val_f1", -1.0)

    loss_fn = make_loss_fn(loss_type, class_weights)

    model = model_builder_fn()
    if encoder_init_state is not None:
        copied = transfer_encoder_weights(encoder_init_state, model)
        print(f"[{label}] Transferred {copied} encoder tensors from SimCSE-pretrained init.")
    model.to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=lr, eps=1e-8)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(WARMUP_FRACTION * total_steps), num_training_steps=total_steps
    )

    best_val_f1 = 0.0
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"[{label}] Epoch {epoch + 1}/{epochs}"):
            input_ids, attn_mask, labels = (t.to(DEVICE) for t in batch)
            model.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
            loss = loss_fn(logits, labels)
            loss.backward()
            total_loss += loss.item()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
        print(f"  [{label}] Epoch {epoch + 1} - train loss: {total_loss / len(train_loader):.4f}")

        model.eval()
        preds, true = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids, attn_mask, labels = (t.to(DEVICE) for t in batch)
                logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
                preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                true.extend(labels.cpu().numpy())
        f1 = f1_score(true, preds, average='micro')
        print(f"  [{label}] Epoch {epoch + 1} - val micro-F1: {f1:.4f}")

        if f1 > best_val_f1:
            best_val_f1 = f1
            patience_counter = 0
            torch.save({"model_state": model.state_dict(), "val_f1": best_val_f1}, save_path)
            print(f"  [{label}] Saved new best checkpoint -> {save_path}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  [{label}] Early stopping.")
                break

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best_val_f1




In [42]:
def enable_mc_dropout(model):
    model.eval()
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()
    return model

In [31]:

def mc_dropout_predict(model, data_loader, n_passes: int = MC_PASSES, device: torch.device = DEVICE):
    model = enable_mc_dropout(model)
    all_pass_probs = []
    true_labels = None

    for pass_idx in tqdm(range(n_passes), desc="  MC Dropout"):
        pass_probs, pass_true = [], []
        with torch.no_grad():
            for batch in data_loader:
                input_ids, attn_mask, labels = (t.to(device) for t in batch)
                logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
                probs = F.softmax(logits, dim=-1)
                pass_probs.append(probs.cpu().numpy())
                if pass_idx == 0:
                    pass_true.extend(labels.cpu().numpy())
        all_pass_probs.append(np.concatenate(pass_probs, axis=0))
        if pass_idx == 0:
            true_labels = np.array(pass_true)

    return np.stack(all_pass_probs), true_labels

In [32]:

def compute_mc_statistics(all_pass_probs, true_labels, label_encoder,
                           n_passes: int = MC_PASSES, confidence: float = CI_CONFIDENCE) -> Dict:
    t_crit = scipy_stats.t.ppf((1 + confidence) / 2, df=n_passes - 1)
    per_pass_preds = np.argmax(all_pass_probs, axis=-1)
    classes = label_encoder.classes_
    n_classes = len(classes)

    per_pass_micro, per_pass_macro, per_pass_per_class = [], [], []
    for p in range(n_passes):
        preds = per_pass_preds[p]
        per_pass_micro.append(f1_score(true_labels, preds, average='micro'))
        per_pass_macro.append(f1_score(true_labels, preds, average='macro'))
        per_pass_per_class.append(f1_score(true_labels, preds, average=None, labels=np.arange(n_classes)))

    per_pass_micro = np.array(per_pass_micro)
    per_pass_macro = np.array(per_pass_macro)
    per_pass_per_class = np.array(per_pass_per_class)

    def ci_tuple(arr):
        m = float(arr.mean())
        s = float(arr.std(ddof=1))
        se = s / np.sqrt(n_passes)
        return m, s, float(m - t_crit * se), float(m + t_crit * se)

    mean_probs = all_pass_probs.mean(axis=0)
    mean_preds = np.argmax(mean_probs, axis=-1)
    pred_labels = label_encoder.inverse_transform(mean_preds)
    true_str = label_encoder.inverse_transform(true_labels)
    clf_rep = classification_report(true_str, pred_labels)

    return {
        'micro': ci_tuple(per_pass_micro),
        'macro': ci_tuple(per_pass_macro),
        'per_class': {cls: ci_tuple(per_pass_per_class[:, i]) for i, cls in enumerate(classes)},
        'clf_report': clf_rep,
    }


In [33]:
def save_results(results_dict: Dict, model_name: str, results_dir: str = RESULTS_DIR,
                  confidence: int = int(CI_CONFIDENCE * 100)):
    slug = model_name.lower().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "").replace("+", "plus")
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    header = f"Model     : {model_name}\nTimestamp : {timestamp}\n"

    with open(f"{results_dir}/{slug}_classification_report.txt", 'w') as f:
        f.write(header + "=" * 60 + "\n")
        f.write("Classification Report (mean prediction across MC passes)\n")
        f.write("=" * 60 + "\n\n" + results_dict['clf_report'])

    with open(f"{results_dir}/{slug}_mc_stats.txt", 'w') as f:
        f.write(header + "=" * 65 + "\n")
        f.write(f"MC Dropout Statistics - {confidence}% CI\n")
        f.write(f"Passes = {MC_PASSES}  |  CI method: t-distribution (df={MC_PASSES - 1})\n")
        f.write("=" * 65 + "\n\n")
        f.write(f"  {'Metric':<22} {'Mean':>8} {'Std':>8} {'CI Lower':>10} {'CI Upper':>10}\n")
        f.write("  " + "-" * 60 + "\n")
        for key, lbl in [('micro', 'Micro-F1'), ('macro', 'Macro-F1')]:
            m, s, lo, hi = results_dict[key]
            f.write(f"  {lbl:<22} {m:>8.4f} {s:>8.4f} {lo:>10.4f} {hi:>10.4f}\n")
        f.write(f"\n  {'Per-Class F1':<22} {'Mean':>8} {'Std':>8} {'CI Lower':>10} {'CI Upper':>10}\n")
        f.write("  " + "-" * 60 + "\n")
        for cls, (m, s, lo, hi) in results_dict['per_class'].items():
            f.write(f"  {cls:<22} {m:>8.4f} {s:>8.4f} {lo:>10.4f} {hi:>10.4f}\n")

    json_payload = {
        'model': model_name, 'timestamp': timestamp, 'mc_passes': MC_PASSES, 'confidence': confidence,
        'micro': dict(zip(['mean', 'std', 'ci_lower', 'ci_upper'], results_dict['micro'])),
        'macro': dict(zip(['mean', 'std', 'ci_lower', 'ci_upper'], results_dict['macro'])),
        'per_class': {cls: dict(zip(['mean', 'std', 'ci_lower', 'ci_upper'], v))
                      for cls, v in results_dict['per_class'].items()},
    }
    with open(f"{results_dir}/{slug}_mc_stats.json", 'w') as f:
        json.dump(json_payload, f, indent=2)
    print(f"  Saved results for '{model_name}' -> {results_dir}/{slug}_*")



In [34]:
def print_mc_report(results: Dict, model_name: str, confidence: int = int(CI_CONFIDENCE * 100)):
    print(f"\n{'=' * 65}\n  {model_name}  -  MC Dropout ({confidence}% CI, {MC_PASSES} passes)\n{'=' * 65}")
    print(f"  {'Metric':<22} {'Mean':>8} {'Std':>8} {'CI Lower':>10} {'CI Upper':>10}")
    for key, lbl in [('micro', 'Micro-F1'), ('macro', 'Macro-F1')]:
        m, s, lo, hi = results[key]
        print(f"  {lbl:<22} {m:>8.4f} {s:>8.4f} {lo:>10.4f} {hi:>10.4f}")
    print(f"\n  {'Per-Class F1':<22} {'Mean':>8} {'Std':>8} {'CI Lower':>10} {'CI Upper':>10}")
    for cls, (m, s, lo, hi) in results['per_class'].items():
        print(f"  {cls:<22} {m:>8.4f} {s:>8.4f} {lo:>10.4f} {hi:>10.4f}")



In [35]:
def non_overlapping_ci_significant(stats_a: Dict, stats_b: Dict, key: str = 'micro') -> bool:

    _, _, a_lo, a_hi = stats_a[key]
    _, _, b_lo, b_hi = stats_b[key]
    return a_hi < b_lo

In [40]:
def run_comparison():
    print(f"\nLoading ground-truth-boundary data from {DATA_DIR} ...")
    train_sents, train_labs, train_prev, train_next = load_pubmed_with_context(f"{DATA_DIR}/train.txt")
    dev_sents, dev_labs, dev_prev, dev_next = load_pubmed_with_context(f"{DATA_DIR}/dev.txt")
    test_sents, test_labs, test_prev, test_next = load_pubmed_with_context(f"{DATA_DIR}/test.txt")
    print(f"Train: {len(train_sents)} | Val: {len(dev_sents)} | Test: {len(test_sents)}")

    label_encoder = LabelEncoder()
    train_labels = label_encoder.fit_transform(train_labs)
    dev_labels = label_encoder.transform(dev_labs)
    test_labels = label_encoder.transform(test_labs)
    num_labels = len(label_encoder.classes_)
    print(f"Classes ({num_labels}): {label_encoder.classes_}")

    counts = np.bincount(train_labels)
    raw_w = 1.0 / counts.astype(float)
    class_weights = torch.tensor(raw_w / raw_w.sum() * num_labels, dtype=torch.float).to(DEVICE)
    print(f"Class weights: {class_weights.cpu().numpy().round(4)}")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    all_results = []  # list of (name, stats_dict) for the final comparison table

    if selected_model in PUBLISHED_CADSCA_RESULTS:
        pub = PUBLISHED_CADSCA_RESULTS[selected_model]
        all_results.append((f"[Published] Single-sentence FT ({selected_model})", pub["baseline"]))
        all_results.append((f"[Published] CA-DSCA ({selected_model})", pub["cadsca"]))
        print(f"\nLoaded published reference numbers for '{selected_model}' from the manuscript "
              f"(no training -- Table IV/VI values, CI recomputed with the same t-distribution formula).")
    else:
        print(f"\nWARNING: no published CA-DSCA reference numbers on file for '{selected_model}'. "
              f"Add an entry to PUBLISHED_CADSCA_RESULTS if you have it, otherwise this run will "
              f"only produce Config 1's own numbers with nothing to compare them against.")

    simcse_state = None
    if RUN_CONFIG1_FOCAL or RUN_CONFIG1_CE:
        simcse_ckpt = f"{SAVE_DIR}/{selected_model}_simcse_encoder.pt"
        if os.path.exists(simcse_ckpt):
            print(f"\nLoading existing SimCSE-pretrained encoder from {simcse_ckpt}")
            simcse_state = torch.load(simcse_ckpt, map_location=DEVICE)
        else:
            print(f"\n{'#' * 78}\n# Config 1: SimCSE contrastive pretraining (shared init)\n{'#' * 78}")
            unlabeled_texts = load_unlabeled_sentences(UNLAB_CSV)
            simcse_encoder = SimCSEModel(BASE_MODEL, dropout=0.2)
            if unlabeled_texts:
                pretrain_simcse(simcse_encoder, unlabeled_texts, tokenizer, device=DEVICE)
            simcse_state = get_base_transformer(simcse_encoder).state_dict()
            torch.save(simcse_state, simcse_ckpt)
            print(f"Saved SimCSE-pretrained encoder -> {simcse_ckpt}")
            del simcse_encoder
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print("\nBuilding Config 1 context strings (prev [SEP] target [SEP] next, ground-truth boundaries)...")
        train_ctx1 = build_config1_context_string(train_sents, train_prev, train_next)
        dev_ctx1 = build_config1_context_string(dev_sents, dev_prev, dev_next)
        test_ctx1 = build_config1_context_string(test_sents, test_prev, test_next)

        print("Encoding Config 1 datasets...")
        train_ds_c1 = encode_single_stream(train_ctx1, train_labels, tokenizer, CONFIG1_MAX_LEN)
        val_ds_c1 = encode_single_stream(dev_ctx1, dev_labels, tokenizer, CONFIG1_MAX_LEN)
        test_ds_c1 = encode_single_stream(test_ctx1, test_labels, tokenizer, CONFIG1_MAX_LEN)
        train_loader_c1 = DataLoader(train_ds_c1, sampler=RandomSampler(train_ds_c1), batch_size=BATCH_SIZE)
        val_loader_c1 = DataLoader(val_ds_c1, sampler=SequentialSampler(val_ds_c1), batch_size=BATCH_SIZE)
        test_loader_c1 = DataLoader(test_ds_c1, sampler=SequentialSampler(test_ds_c1), batch_size=BATCH_SIZE)

        for run_flag, loss_type, name in [
            (RUN_CONFIG1_FOCAL, "focal", f"Config 1 (SimCSE + context, Focal) [{selected_model}]"),
            (RUN_CONFIG1_CE, "ce", f"Config 1 (SimCSE + context, CE) [{selected_model}]"),
        ]:
            if not run_flag:
                continue
            ckpt_path = f"{SAVE_DIR}/{selected_model}_config1_{loss_type}.pt"
            print(f"\n{'#' * 78}\n# {name}\n{'#' * 78}")
            train_single_stream(
                model_builder_fn=lambda: AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=num_labels),
                num_labels=num_labels, train_loader=train_loader_c1, val_loader=val_loader_c1,
                save_path=ckpt_path, class_weights=class_weights, loss_type=loss_type,
                encoder_init_state=simcse_state, label=name
            )
            model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=num_labels)
            model.load_state_dict(torch.load(ckpt_path, map_location='cpu')["model_state"])
            model.to(DEVICE)
            probs, true = mc_dropout_predict(model, test_loader_c1)
            stats = compute_mc_statistics(probs, true, label_encoder)
            print_mc_report(stats, name)
            save_results(stats, name)
            all_results.append((name, stats))
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


In [ ]:
if __name__ == "__main__":
    run_comparison()


Loading ground-truth-boundary data from /content/drive/MyDrive/Zhi_folders/PubMed/pubmed-rct-master/PubMed_20k_RCT ...
Train: 180040 | Val: 30212 | Test: 30135
Classes (5): ['BACKGROUND' 'CONCLUSIONS' 'METHODS' 'OBJECTIVE' 'RESULTS']
Class weights: [1.2163 0.9727 0.4453 1.9096 0.456 ]

Loaded published reference numbers for 'pubmedbert' from the manuscript (no training -- Table IV/VI values, CI recomputed with the same t-distribution formula).

Loading existing SimCSE-pretrained encoder from /content/drive/MyDrive/Zhi_folders/config1_vs_cadsca/pubmedbert/pubmedbert_simcse_encoder.pt

Building Config 1 context strings (prev [SEP] target [SEP] next, ground-truth boundaries)...
Encoding Config 1 datasets...

##############################################################################
# Config 1 (SimCSE + context, Focal) [pubmedbert]
##############################################################################
[Config 1 (SimCSE + context, Focal) [pubmedbert]] Checkpoint already exists 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

  MC Dropout:   0%|          | 0/30 [00:00<?, ?it/s]